# Quy trình đánh giá chéo gán nhãn giữa con người và LLM
**Mục tiêu**: Đánh giá chéo để kiểm chứng hiệu quả gán nhãn giữa người với người, giữa người và LLM

In [7]:
from pathlib import Path
import pandas as pd
import re
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa
from sklearn.metrics import classification_report, accuracy_score

PROJECT_ROOT = Path.cwd().parent.parent.resolve()

GOLD_FEATURE = PROJECT_ROOT / "Data" / "Gold_layer" / "Features"
AUDIT = PROJECT_ROOT / "Data" / "Gold_layer" / "Audit"

## Tách validation set để đánh giá chéo

In [3]:
INPUT_ORIGINAL_FILE = GOLD_FEATURE / "master_departure_features_gold.csv"
OUTPUT_SPLIT_FILE = GOLD_FEATURE / "validation_departure_features_gold.csv"
df_sample = pd.read_csv(INPUT_ORIGINAL_FILE)

# Đọc file kết quả sau khi chạy LLM
df = pd.read_csv(INPUT_ORIGINAL_FILE)

# Chỉ lọc các chuyến bay trễ từ 15 phút trở lên
df_delay = df[df['Departure_Delay'] >= 15].copy()

# Lấy ngẫu nhiên một tập mẫu để đánh giá chéo
df_sample = df_delay.sample(n=min(100, len(df_delay)), random_state=42)

# Chuyển đổi các thông số số liệu thành một đoạn văn bản trực quan để hiển thị trên Doccano
def make_readable_text(row):
    text = (
        f"Chuyến bay: {row.get('Flight_No', 'Unknown')} | Thời gian trễ: {row.get('Departure_Delay', 0)} phút.\n"
        f"----------------------------------------\n"
        f"- Quỹ thời gian quay đầu (Turnaround Buffer): {row.get('Turnaround_Buffer', 0)} phút.\n"
        f"- Thời gian trễ tích lũy từ chặng trước (Accumulated Delay): {row.get('Accumulated_Delay', 0)} phút.\n"
        f"- Chỉ số rủi ro thời tiết tại sân bay đi (Weather Risk Score): {row.get('Weather_Delay_Risk_Score', 0)}.\n"
        f"- Tải lượng sân bay đi (Airport Load Factor): {row.get('Airport_Load_Factor', 0)}.\n"
        f"- Nguy cơ tắc nghẽn tại điểm đến (Destination Congestion Risk): {row.get('Destination_Congestion_Risk', 0)}.\n"
        f"- Áp dụng bãi đỗ xa (Is Remote Stand): {'Có (1)' if row.get('Is_Remote_Stand')==1 else 'Không (0)'}.\n"
        f"- Nguy cơ tắc nghẽn do không lưu (Taxi_Out_Congestion): {row.get('Taxi_Out_Congestion', 0)}.\n"
    )
    return text

df_sample['Flight Information'] = df_sample.apply(make_readable_text, axis=1)
df_sample['LLM_Delay_Code'] = ''

# Xuất ra file Excel hoặc CSV chỉ giữ lại cột 'text' và ID gốc để import vào Doccano
df_sample[['Flight_No', 'Scheduled_Time', 'Flight Information', 'LLM_Delay_Code']].to_csv(OUTPUT_SPLIT_FILE, index=False)
print('[V] Đã trích xuất validation set thành công.')

[V] Đã trích xuất validation set thành công.


## Tính độ đồng thuận gãn nhãn giữa 3 người

In [4]:
# BƯỚC 1: ĐỌC VÀ ĐỔI TÊN CỘT TỪ 3 FILE CSV
ADMIN_PATH = GOLD_FEATURE / "annotated_validation_set" / "admin.csv"
ANNOTATOR1_PATH = GOLD_FEATURE / "annotated_validation_set" / "hunf299.csv"
ANNOTATOR2_PATH = GOLD_FEATURE / "annotated_validation_set" / "huong0811.csv"
OUTPUT_CONSENSUS_FILE = GOLD_FEATURE / "human_annotated_results.csv"

df_admin = pd.read_csv(ADMIN_PATH)
df_annotator1 = pd.read_csv(ANNOTATOR1_PATH)
df_annotator2 = pd.read_csv(ANNOTATOR2_PATH)

# Giữ lại ID, Text và đổi tên cột nhãn để dễ phân biệt
df_admin = df_admin.rename(columns={'label': 'admin_label'})
df_annotator1 = df_annotator1.rename(columns={'label': 'hunf_label'})
df_annotator2 = df_annotator2.rename(columns={'label': 'huong_label'})

# BƯỚC 2: GỘP DỮ LIỆU BẰNG ID HOẶC TEXT
df_merged = df_admin[['id', 'text', 'admin_label']].merge(
    df_annotator1[['id', 'hunf_label']], on='id'
).merge(
    df_annotator2[['id', 'huong_label']], on='id'
)

# BƯỚC 3: TÍNH ĐỘ ĐỒNG THUẬN FLEISS' KAPPA
# Rút trích ma trận chứa kết quả của 3 người
ratings_matrix = df_merged[['admin_label', 'hunf_label', 'huong_label']].values

# Hàm aggregate_raters chuyển đổi ma trận sang dạng đếm tần suất nhãn cho hàm fleiss_kappa
agg_ratings, _ = aggregate_raters(ratings_matrix)
kappa = fleiss_kappa(agg_ratings, method='fleiss')

print("--- ĐÁNH GIÁ ĐỘ ĐỒNG THUẬN GIỮA 3 NGƯỜI ---")
print(f"Chỉ số Fleiss' Kappa: {kappa:.3f}")
if kappa > 0.75:
    print("-> Kết luận: Độ đồng thuận RẤT TỐT!")
elif kappa >= 0.40:
    print("-> Kết luận: Độ đồng thuận TRUNG BÌNH.")
else:
    print("-> Kết luận: Độ đồng thuận KÉM.")

# BƯỚC 4: TẠO CHUẨN VÀNG BẰNG MAJORITY VOTE
def get_majority_vote(row):
    # Gom 3 phiếu bầu lại
    votes = [row['admin_label'], row['hunf_label'], row['huong_label']]
    vote_counts = pd.Series(votes).value_counts()

    # Nếu có nhãn chiếm >= 2 phiếu (tức là 2/3 người hoặc cả 3 người cùng chọn)
    if vote_counts.iloc[0] >= 2:
        return vote_counts.index[0]
    else:
        # Trường hợp hy hữu: Cả 3 người chọn 3 mã lỗi IATA KHÁC NHAU hoàn toàn
        return 'CONFLICT'

df_merged['Ground_Truth'] = df_merged.apply(get_majority_vote, axis=1)

conflicts = df_merged[df_merged['Ground_Truth'] == 'CONFLICT']
print(f"\nSố lượng chuyến bay cần họp Trọng tài (3 người ra 3 mã khác nhau): {len(conflicts)}")
if len(conflicts) > 0:
    print("Vui lòng xem các ID sau và thảo luận chốt lại nhãn:")
    print(conflicts[['id', 'admin_label', 'hunf_label', 'huong_label']])

df_merged.to_csv(OUTPUT_CONSENSUS_FILE, index=False, encoding='utf-8-sig')
print("\nĐã xuất file tổng hợp thành công: human_annotated_results.csv")

--- ĐÁNH GIÁ ĐỘ ĐỒNG THUẬN GIỮA 3 NGƯỜI ---
Chỉ số Fleiss' Kappa: 0.574
-> Kết luận: Độ đồng thuận TRUNG BÌNH.

Số lượng chuyến bay cần họp Trọng tài (3 người ra 3 mã khác nhau): 6
Vui lòng xem các ID sau và thảo luận chốt lại nhãn:
      id admin_label hunf_label huong_label
17  1119     CODE_81    CODE_89     CODE_99
27  1129     CODE_81    CODE_89     CODE_99
34  1136     CODE_89    CODE_06     CODE_99
62  1164     CODE_99    CODE_81     CODE_89
68  1170     CODE_99    CODE_89     CODE_81
98  1200     CODE_99    CODE_89     CODE_81

Đã xuất file tổng hợp thành công: human_annotated_results.csv


## Thống nhất nhãn bị xung đột giữa 3 annotators

In [17]:
# 1. Đọc file kết quả từ bước trước
df_human = pd.read_csv(OUTPUT_CONSENSUS_FILE)

# 2. CẬP NHẬT KẾT QUẢ SAU THỐNG NHẤT
adjudicated_labels = {
    1119: 'CODE_99',
    1129: 'CODE_99',
    1136: 'CODE_99',
    1164: 'CODE_99',
    1170: 'CODE_99',
    1200: 'CODE_99'
}

# Áp dụng nhãn đã phân xử vào cột Ground_Truth
for id_flight, final_label in adjudicated_labels.items():
    df_human.loc[df_human['id'] == id_flight, 'Ground_Truth'] = final_label

df_human.to_csv(OUTPUT_CONSENSUS_FILE, index=False, encoding='utf-8-sig')

## Đánh giá chéo kết quả của LLM và con người

In [18]:
# 3. ĐỌC FILE KẾT QUẢ CỦA LLM
INPUT_LLM_FILE = GOLD_FEATURE / "master_departure_features_gold_annotated.csv"
OUTPUT_EVALUATION_FILE = AUDIT / "final_cross_evaluation_report.csv"
df_llm = pd.read_csv(INPUT_LLM_FILE, dtype={'LLM_Delay_Code': str}, low_memory=False)

df_llm = df_llm[~df_llm['LLM_Delay_Code'].isin(['-1', '-1.0'])].copy()

def extract_flight_info(text):
    # Tìm chuỗi có chữ "Chuyến bay:" và lấy mã phía sau
    flight_match = re.search(r'Chuyến bay: ([\w\d]+)', str(text))
    # Tìm số phút trễ
    delay_match = re.search(r'Thời gian trễ: ([\d\.]+) phút', str(text))

    return pd.Series({
        'Flight_No': flight_match.group(1) if flight_match else None,
        'Departure_Delay': float(delay_match.group(1)) if delay_match else None
    })

# Tách thông tin và tạo 2 cột mới trong bảng dữ liệu của con người
df_human[['Flight_No', 'Departure_Delay']] = df_human['text'].apply(extract_flight_info)

# 4. Tiến hành Map (Merge) 2 bảng lại với nhau
df_final = df_human.merge(
    df_llm[['Flight_No', 'Departure_Delay', 'LLM_Delay_Code']],
    on=['Flight_No', 'Departure_Delay'],
    how='left'
)

# Xóa dòng lặp và điền nhãn "MISSING" nếu có chuyến nào bị rớt trong lúc map
df_final = df_final.drop_duplicates(subset=['id']).reset_index(drop=True)
df_final['LLM_Delay_Code'] = df_final['LLM_Delay_Code'].fillna('MISSING')

# 5. XỬ LÝ CA CONFLICT (HỌP TRỌNG TÀI)
df_eval = df_final[df_final['Ground_Truth'] != 'CONFLICT'].copy()

# 6. IN BÁO CÁO ĐÁNH GIÁ CHÉO
print("==========================================================================")
print("     BÁO CÁO ĐÁNH GIÁ CHÉO: CON NGƯỜI VS LOCAL LLM")
print("==========================================================================")
print(f"Tổng số mẫu đưa vào đối chiếu: {len(df_eval)} chuyến bay (đã loại trừ ca CONFLICT)\n")

acc = accuracy_score(df_eval['Ground_Truth'], df_eval['LLM_Delay_Code'])
print(f"Độ chính xác tổng quan (Accuracy): {acc*100:.2f}%\n")

print("Chi tiết hiệu suất gán nhãn của LLM trên từng mã lỗi IATA:")
print(classification_report(
    df_eval['Ground_Truth'],
    df_eval['LLM_Delay_Code'],
    labels=['CODE_71', 'CODE_81', 'CODE_89', 'CODE_06', 'CODE_93', 'CODE_99'],
    zero_division=0
))

# Xuất file kết quả hoàn chỉnh
df_final.to_csv(OUTPUT_EVALUATION_FILE, index=False, encoding='utf-8-sig')
print("\n[V] Đã xuất file báo cáo tổng hợp: final_cross_evaluation_report.csv")

     BÁO CÁO ĐÁNH GIÁ CHÉO: CON NGƯỜI VS LOCAL LLM
Tổng số mẫu đưa vào đối chiếu: 100 chuyến bay (đã loại trừ ca CONFLICT)

Độ chính xác tổng quan (Accuracy): 26.00%

Chi tiết hiệu suất gán nhãn của LLM trên từng mã lỗi IATA:
              precision    recall  f1-score   support

     CODE_71       0.03      1.00      0.06         1
     CODE_81       0.00      0.00      0.00         3
     CODE_89       0.59      0.57      0.58        40
     CODE_06       1.00      1.00      1.00         1
     CODE_93       0.00      0.00      0.00         0
     CODE_99       1.00      0.02      0.04        55

   micro avg       0.28      0.26      0.27       100
   macro avg       0.44      0.43      0.28       100
weighted avg       0.80      0.26      0.26       100


[V] Đã xuất file báo cáo tổng hợp: final_cross_evaluation_report.csv
